In [ ]:

import polars as pl
import polars.selectors as cs
from project_paths import paths

df = pl.read_parquet(paths.processed_data / "unified_aims_eir_bgs.parquet")

df.columns

In [ ]:
context = ["aims__asset_id", "aims__asset_sub_type", "aims__water_course_name"]

lengths = ["aims__asset_length"]
levels = [
    "aims__toe_level", 
    "aims__actual_dcl", 
    "aims__actual_ucl",
    "aims__design_dcl", 
    "aims__design_ucl", 
    "aims__effective_cl"
]
sops = ["aims__current_sop", "aims__design_sop"]

numeric = lengths + levels + sops

In [ ]:
df.select(numeric).describe()

In [ ]:
df.select(context + ["aims__asset_length"]).sort("aims__asset_length", descending=True).head(10)

In [ ]:
df.select(context + ["aims__asset_length"]).sort("aims__asset_length").head(10)

there is obviously going to be a strong relationship / amount of mutual information between the asset type / subtype and the length. Probably also other relationships between the type/subtype and other physical design criteria, like the crest levels. 

In terms of data validity here, I was looking for obviously wrong values such as something being exactly a huge or tiny number, or an impossible number (0 or negative). 

In [ ]:
for field in numeric:
    print(df.select(context + [field]).sort(field, descending=True, nulls_last=True).head(10))
    print(df.select(context + [field]).sort(field).head(10))
    print("\n\n")

In [ ]:
warning_values = [0, -1, 999, 9999, 99999, -99, -999, -9999, -99999]

df.select(pl.col(numeric).is_in(warning_values).sum().name.suffix("__n_warn"))

In [ ]:

with pl.Config(tbl_rows=-1, tbl_cols=-1, tbl_width_chars=220):
    print(
        df.filter(pl.any_horizontal(pl.col(n).is_in(warning_values) for n in numeric))
        .select(context + numeric)
    )


toe level contains both 0s and nulls, and can have nulls and positive values for the same asset type. Null could mean not measured? There is also one with minus one as the value - this could actually be correct if the water level is a metre below the main watercouse level/sea level or something like that...

however asset id 8154 has a design dcl of -9999 and this seems like an obvious impossiblility. 



In [ ]:
df.select(
    (pl.col(lengths) <= 0).sum().name.suffix("__n_nonpositive"),
    (pl.col(sops) <= 0).sum().name.suffix("__n_nonpositive"),
    (pl.col("aims__asset_length") > 5000).sum().alias("length__n_over_5km"),
)

In [ ]:
with pl.Config(tbl_rows=-1, tbl_cols=-1, tbl_width_chars=220):
    print(
        df.filter(pl.any_horizontal(( pl.col(n).is_in(warning_values) | pl.col(n).lt(0)) for n in numeric))
        .select(context + numeric)
    )

based on this output, I think that the -1 value for toe level is genuine and should stay in. 

The only value that I think is a clearly invalid input is the -9999 value for design dcl. Given its a single row, I would be happy to drop it and continue. However, I found in the previous notebooks that all of the crest level features are extremely strongly corelated. The best solution will be to determine which crest level feature has to best coverage and integrety and just use that one. 


The more complicated bit is that I think there are multiple conventions used here for null/missing/not measured, including the actual nulls, the 0s, and the impossible numbers like -9999 as well as some vlues that I think are appearing too often to be genuine like 99. It could be that this data is an amalgimation of different forms filled in by different people/orgs that use a different convention. 

For example, i can see 0 for crest level in some rows where the toe level is recorded as positive. I think its impossible for the crest to be higher than the toe level. In these cases, 0 probably means not measured or not applicable in some way. 

I think a similar thing about design crest level being 0 when actual crest level is positive. 

It also looks like the 99s in design downstream crest level are part of a repeated group where the asset is an embankment, where the design ucl is null, the toe level is 0 or near 0, and the actual dcl is ~5m. I dont know why 99 occurs in this pattern, but the presence of the pattern suggests to me that this could be a batch fill where 99 is a placeholder of some kind. Basically, i think its incredibly unlikely for actual crest level to be around 5m if the design was genuinely 99m. Also design sop is 200 for all of these rows.


Overall, its good news that there are such a small number of rows with dubious inputs considering the size of the dataset - 66 rows to look at out of ~35k with some that look genuine, around 30 need correcting. Unlikely to actually skew model results much even if left in. 

Decision - any rows that I clean, I will set the values to null rather than impute. 



In [ ]:
measured_crest = ["aims__actual_dcl", "aims__actual_ucl", "aims__effective_cl"]

crest_below_toe = (
    df.filter(
        pl.any_horizontal(pl.col(c) < pl.col("aims__toe_level") for c in measured_crest)
    )
    .select(context + ["aims__toe_level"] + measured_crest)
)


with pl.Config(tbl_rows=-1, tbl_cols=-1, tbl_width_chars=220):
    print(
        crest_below_toe
    )

when writing this test i was intending it to catch physical impossibilities, but looking at the results, is it possible that its true for flood gates on tidal rivers?
A lot of the numbers here look genuine - no placeholders, suspiciously round numbers, 0s, etc. So could it just be a measurement or input error? or measurements taken at different times? 

In [ ]:
threshold = 2.0

design_actual_gap = (
    df.with_columns(
        (pl.col("aims__design_dcl") - pl.col("aims__actual_dcl")).abs().alias("dcl_gap"),
        (pl.col("aims__design_ucl") - pl.col("aims__actual_ucl")).abs().alias("ucl_gap"),
    )
    .filter((pl.col("dcl_gap") > threshold) | (pl.col("ucl_gap") > threshold))
    .select(
        context
        + [
            "aims__design_dcl", 
            "aims__actual_dcl", 
            "dcl_gap",
            "aims__design_ucl", 
            "aims__actual_ucl", 
            "ucl_gap"
        ]
    )
    .sort("dcl_gap", descending=True, nulls_last=True)
)

with pl.Config(tbl_rows=-1, tbl_cols=-1, tbl_width_chars=220):
    print(
        design_actual_gap
    )

some possiblilities:
    - 100 and 0.001 are also placeholders for invalid inputs
    - 100 and 0.001 are inputs in the incorrect units (i.e. 100 cm instead of 1 metre, would explain most of these)

almost all of these cases occur in downstream crest design/measurement, not the upstream. 


Its still a reletively low number of cases. This has two implications:
    1. not a big deal if we have to null/exclude these cases, except that it might systematically be distorting coverage to Embankment asset types
    2. I was starting to think that design to actual gap would be an interesting feature - it makes sense that assets that have to operate outside of the conditions that they were designed for may experience stresses they weren't expected to see and may degrade faster (or slower perhaps)



decisions:

    - set to null:
        - 100 values and 99 values and -9999 value in aims__design_dcl with significant gap to actual__dcl
        - 

Decisions:

    - set to null:
        - implausable values in aims__design_dcl: 
            - -9999
            - 100 and 99, 0.001, 18900 where there is a significant gap to actual dcl
            - keep values when there is a small gap to actual dcl since all positive numbers are not pysically impossible inputs to the field

    - leave all toe level values, all seem genuine.
    - leave all asset lengths - checked all over 5k and seem realistic for asset type
    - 

    - return to actual_dcl before modelling
        - it is the likeliest candidate for the crest feature kept for modelling - best coverage
        - implausible values - eight rows have 0s, same gap logic as described above

    - SOP columns, not sure if keeping but if keeping design sop then null a 0 and a 9999.  

        